# Semantic shortcuts

**Rung 3 to 5 of 5 (next: rung 4, [09, Scale](09_scale.ipynb)). You are here: the same
addresses, now spelled in the words of the architecture.**
Trigger: you keep typing "head 3's query".

Everything so far addressed a model by its labels and module names, and that works for every
model there is. Attention models have a second vocabulary: query, key, value, head, pattern.
TorchLens calls these facets. A facet is a named view over an op you already have; you can teach
TorchLens the words for your own model, and Hugging Face models come with the words already
known. The last section teaches TorchLens a return type it has never seen.

> **You'll want to know:** [module addresses](02_address_and_read.ipynb) (`trace["q_proj"]` is
> that module's output), [`intervene=`](05_change_and_rerun.ipynb) (an edit at a selected site),
> [containers](02_address_and_read.ipynb) (a model that returns a dict keeps the dict).

In [ ]:
import tempfile
from dataclasses import dataclass
from pathlib import Path

import torch
from IPython.display import display
from IPython.utils.capture import capture_output
from torch import nn

import torchlens as tl

PICTURES = Path(
    tempfile.mkdtemp(prefix="torchlens_learn_")
)  # drawings land here, not beside the notebook

In [ ]:
from learn_models import TinyAttention, tiny_attention, tiny_mlp  # tiny seeded CPU models

model, ids = (
    tiny_attention()
)  # one attention layer: q_proj, k_proj, v_proj, two heads; a (1, 5) batch of ids

The long way first: head 1's query, by module name and a reshape you do yourself. The rest of the
notebook turns this into `facets.head(1).q`.

In [ ]:
trace = tl.trace(model, ids)
q = trace["q_proj"].out  # (batch, positions, width)
by_hand = q.view(1, 5, model.heads, -1)[:, :, 1]  # split the width into heads, keep head 1
print(tuple(q.shape), "->", tuple(by_hand.shape))

## 1. The long way works for every model

A projection is one linear layer, so its output is one op with one label. Splitting it into heads
is a reshape, and "head 1's query" is a slice of that reshape. Nothing here is attention
specific, which is the point: the addresses from 02 reach every tensor a model makes.

In [ ]:
k = trace["k_proj"].out
v = trace["v_proj"].out
print(
    "labels:", trace["q_proj"].layer_label, trace["k_proj"].layer_label, trace["v_proj"].layer_label
)
print(
    "one head of k:",
    tuple(k.view(1, 5, model.heads, -1)[:, :, 1].shape),
    "| of v:",
    tuple(v.view(1, 5, model.heads, -1)[:, :, 1].shape),
)

## 2. Hugging Face models already speak it

For the model families TorchLens ships recipes for, nothing is registered by hand: the facets
are there on the first trace. This cell builds a two-layer GPT-2 from a config with random
weights (no download) and reads one block's attention; eager attention keeps the softmax pattern
as a real op (a fused kernel hides it, and the facet says so). With a real checkpoint and its
tokenizer, `tl.trace(lm, "a prompt")` takes the string as the input; nothing here fetches one.
If that is all you need, you can stop here; the next section is for a model of your own.


In [ ]:
try:
    import transformers
except ImportError:
    transformers = None
    print(
        "transformers is not installed; this cell is skipped (pip install transformers to run it)"
    )

if transformers is not None:
    config = transformers.GPT2Config(
        n_layer=2,
        n_head=2,
        n_embd=32,
        vocab_size=64,
        bos_token_id=0,
        eos_token_id=0,
        attn_implementation="eager",
    )
    lm = transformers.GPT2LMHeadModel(config).eval()
    tokens = torch.tensor([[1, 5, 9, 2]])
    lm_trace = tl.trace(lm, tokens)
    attn = lm_trace.modules["transformer.h.0.attn"].facets
    print("facets:", sorted(attn.keys())[:12])
    print("head 1 q:", tuple(attn.head(1).q.shape), "| pattern:", tuple(attn.pattern.shape))

## 3. Teach TorchLens the words

A recipe is a function that takes a module record and returns named views. `FacetSpec.from_home`
anchors a view to an op, `heads(n, d)` records the split, and the registration names the class
the recipe is for. The recipe reads the head count from the reshape the model itself did (the
`view` op after each projection), so it holds for every `TinyAttention`, whatever its width or
head count, and never looks at a notebook variable. From then on every trace of that class
answers in the new vocabulary, next to the structural facets TorchLens lists for any module
(`out`, weights), and the by-hand slice and the facet agree exactly.


In [ ]:
@tl.facets.register(
    class_name="TinyAttention", target_scope="module", facets=("q", "k", "v", "n_heads")
)
def tiny_attention_facets(module):
    """Name the query, key and value projections of TinyAttention, split into heads."""
    prefix = "" if module.address == "self" else f"{module.address}."  # "self" is the root module

    def projection(name):
        op = module.trace[module.trace.modules[prefix + name].output_layers[0]]
        split = module.trace[op.children[0]]  # the view that splits the width into heads
        n_heads, d_head = split.shape[-2], split.shape[-1]
        return tl.facets.FacetSpec.from_home(op).heads(n_heads, d_head), n_heads

    q, n_heads = projection("q_proj")
    return {"q": q, "k": projection("k_proj")[0], "v": projection("v_proj")[0], "n_heads": n_heads}


trace = tl.trace(model, ids)
facets = trace.modules["self"].facets  # "self" is the root module's address
print("facets:", sorted(facets.keys()))
print("q:", tuple(facets.q.shape), "| head 1 of q:", tuple(facets.head(1).q.shape))
print("same as by hand:", torch.equal(facets.head(1).q.value, by_hand))  # .value is the tensor

wider = tl.trace(TinyAttention(width=16, heads=4).eval(), ids)  # another configuration, same recipe
print(
    "a 4-head instance:",
    wider.modules["self"].facets.n_heads,
    "heads | head 3 of q:",
    tuple(wider.modules["self"].facets.head(3).q.shape),
)

## 4. Heads as selectors

The words also select. `tl.head(1, "q")` names head 1 of the query facet. A facet is resolved on a
finished trace, so the edit goes through `fork().do(where, what)` from 05 rather than `intervene=`:
head 1 is zeroed, head 0 is untouched, and the downstream values replay. The by-hand route would
have needed a hook that reshapes, slices and writes back.

In [ ]:
ready = tl.trace(model, ids)
muted = ready.fork("mute_head_1")
with capture_output():  # the replay draws a progress bar for its 13 sites; nothing else to see
    muted.do(tl.head(1, "q"), tl.zero_ablate())
after = muted.modules["self"].facets
print(
    "head 1 q energy:",
    round(float(after.head(1).q.value.detach().abs().sum()), 3),
    "| head 0:",
    round(float(after.head(0).q.value.detach().abs().sum()), 3),
)
print(
    "output moved by:",
    round(float((muted.reconstruct_output() - ready.reconstruct_output()).detach().norm()), 3),
)

## 5. Custom return types

02 showed that a model returning a dict keeps the dict. A type TorchLens has never seen needs
two small functions: how to take it apart into tensors and how to put it back together.
`tl.types.register_container` records them, and from then on `trace.reconstruct_output()` returns your type again.

In [ ]:
@dataclass
class Scored:
    logits: torch.Tensor
    confidence: torch.Tensor


class ScoredMLP(nn.Module):
    def __init__(self, inner):
        super().__init__()
        self.inner = inner

    def forward(self, x):
        logits = self.inner(x)
        return Scored(logits, logits.softmax(-1).max(-1).values)


tl.types.register_container(
    Scored,
    lambda scored: (
        [scored.logits, scored.confidence],
        None,
    ),  # take it apart: children and context
    lambda context, children: Scored(*children),  # put it back together
)

mlp, x = tiny_mlp()
scored_trace = tl.trace(ScoredMLP(mlp), x)
print(
    type(scored_trace.reconstruct_output()).__name__,
    "| confidence:",
    scored_trace.reconstruct_output().confidence.detach(),
)
print("output layers:", scored_trace.output_layers)

## 6. That was the vocabulary rung

`module.facets` reads named views, `facets.head(i)` slices them, `tl.head(i, "q")` selects them
for an edit, `tl.facets.register` teaches the words for a new class, and
`tl.types.register_container` teaches a new return type. The labels from 02 never stopped
working underneath.

**One thing about networks:** q, k and v are one linear layer each; a "head" is a view of a
reshape, not a module, and no tensor in the forward pass is ever called `head_1`. Facets name
views, which is why they work on any model once told where to look.

**Go deeper:** next is [09, Scale](09_scale.ipynb). Workflows:
[transformer_migration](../../examples/50min/transformer_migration.ipynb) (TransformerLens hook
names as facets) and [paired_prompt_patching](../../examples/50min/paired_prompt_patching.ipynb).
Reference: [the mechinterp kit](../../docs/reference/mechinterp.md) and the glossary's "Facet"
and "Container registration" entries in [glossary.md](../../docs/reference/glossary.md).